# MCP server basics — three tools, fully wired

The Model Context Protocol is JSON-RPC 2.0 with a small set of canonical methods (`initialize`, `tools/list`, `tools/call`, `resources/*`, `prompts/*`). This notebook walks a minimal server end-to-end so you can see what FastMCP/`mcp-python` wraps.

## Production code — paste into a project with `fastmcp`

```python
from fastmcp import FastMCP

mcp = FastMCP('corpus-server')

@mcp.tool()
def search_corpus(query: str, k: int = 3) -> list[dict]:
    """Hybrid search over the canonical arxiv corpus."""
    return search(query, k)

@mcp.tool()
def fetch_paper(arxiv_id: str) -> dict:
    """Return the full record for an arxiv id."""
    return _DOC_BY_ID[arxiv_id].model_dump()

if __name__ == '__main__':
    mcp.run(transport='stdio')
```

Below we use the in-repo `mcp_core` server so the notebook runs offline.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / '06-mcp'))
os.environ.setdefault('LLM_CACHE_ONLY', '1')

from mcp_core import build_corpus_server, Client, InProcessTransport

server = build_corpus_server()
client = Client(transport=InProcessTransport(server))
client.initialize()['serverInfo']

In [ ]:
for t in client.list_tools():
    print(t['name'].ljust(16), '->', t['description'])
    print('     schema   :', t['inputSchema'])

In [ ]:
hits = client.call_tool('search_corpus', {'query': 'mixture of experts', 'k': 2})
print('hits  :', [h['arxiv_id'] for h in hits])
paper = client.call_tool('fetch_paper', {'arxiv_id': hits[0]['arxiv_id']})
print('title :', paper['title'])
citation = client.call_tool('cite', {'arxiv_id': hits[0]['arxiv_id'], 'claim': paper['abstract'][:80]})
print('cite  :', citation)
print('audit :', [e['method'] for e in server.audit_log])

## What the in-process transport hides

In production the client speaks JSON over stdio/SSE/websocket to a child process. Swapping `InProcessTransport` for a stdio transport is a one-line change — `tools/list` and `tools/call` look identical to the agent.